# RFI explorer

Data | data - DPSS smooth model | kept-only residual (flags/v2), for one
antenna over one window, straight from `eigsep_data`: the metadata index
picks the rows, `load_bundle` joins the raw counts to their `flags@v2` and
`smooth_model@v0` companions on a common time/frequency axis.

No widgets: cell 2 prints the filespans available, cell 3 plots the one
you name in `SPAN` (and `MINUTES`, `ANTENNA`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from eigsep_data import MetadataIndex

CAMPAIGN = "/mnt/data02/eigsep/marjum-2026-07"
BAND_MHZ = (45.0, 235.0)
PRODUCTS = ["flags@v2", "smooth_model@v0"]

idx = MetadataIndex(f"{CAMPAIGN}/data")
sel = idx.select()
print(sel.summary())
%matplotlib widget

## Filespans available

Contiguous stretches of observing, split wherever the data stops for more
than `GAP_S`. Times are `time_best` (header time where the clock was sane,
filename estimate otherwise) -- not the filename close times.

In [ ]:
GAP_S = 600

meta = sel.meta.assign(span=sel.visits(gap_s=GAP_S))
spans = pd.DataFrame(
    [
        dict(
            span=sid,
            start=pd.to_datetime(g.time_best.min(), unit="s"),
            end=pd.to_datetime(g.time_best.max(), unit="s"),
            hours=(g.time_best.max() - g.time_best.min()) / 3600,
            files=g.file.nunique(),
            rows=len(g),
            first_file=g.file.iloc[0],
            last_file=g.file.iloc[-1],
        )
        for sid, g in meta.groupby("span")
    ]
).set_index("span")

with pd.option_context("display.width", 200, "display.max_colwidth", 30):
    print(spans)

## Load and plot

Pick a span, how many minutes from its start to read, and an antenna.
Any window works, not just a whole span -- `idx.select(time=(lo, hi))`
takes real times, and `idx.select(files=(first, last))` takes an
inclusive filename range if that is what you want instead.

In [ ]:
SPAN = 14
MINUTES = 20
ANTENNA = "box-air"   # or "box-gnd"

t0 = spans.loc[SPAN, "start"].timestamp()
t1 = spans.loc[SPAN, "end"].timestamp()
#window = idx.select(time=(t0, t0 + MINUTES * 60))
window = idx.select(time=(t1 - MINUTES * 60, t1))
b = window.load_bundle(antenna=ANTENNA, products=PRODUCTS, band_mhz=BAND_MHZ)
print(b.summary())

In [ ]:
residual = b.residual                      # data - model, recomputed
kept = np.where(b.flags == 0, residual, np.nan)   # v2: any bit set is bad
minutes = (b.t - b.t.min()) / 60
extent = [b.freqs_mhz.min(), b.freqs_mhz.max(), minutes.max(), minutes.min()]

vmax = np.nanpercentile(np.abs(residual), 99)
sym = mcolors.SymLogNorm(linthresh=vmax / 50, vmin=-vmax, vmax=vmax)
data = np.maximum(b.data, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
panels = [
    (data, dict(norm=mcolors.LogNorm(vmin=np.nanmin(data), vmax=np.nanmax(data)),
                cmap="viridis"), "data (counts)"),
    (residual, dict(norm=sym, cmap="RdBu_r"), "data - DPSS model"),
    (kept, dict(norm=sym, cmap="RdBu_r"), "residual, flags/v2 kept only"),
]
for ax, (arr, kw, title) in zip(axes, panels):
    im = ax.imshow(arr, aspect="auto", extent=extent, **kw)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Frequency (MHz)")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
t_start = spans.loc[SPAN, "start"].strftime("%Y-%m-%d %H:%M:%SZ")
axes[0].set_ylabel(f"minutes from {t_start}")
fig.suptitle(f"span {SPAN}, {ANTENNA}, {MINUTES} min", fontsize=10)
fig.tight_layout()
plt.show()